# BeatNet-CRNN - Stima del BPM di un brano

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alessandro1040/bpm-crnn/blob/main/notebooks/01_demo_stima_bpm.ipynb)

Questo notebook carica il modello **gia' addestrato** (pesi pubblicati in
`models/bpm_crnn_gtzan.pt` di questo repo e nella
[release v1.0](https://github.com/Alessandro1040/bpm-crnn/releases/tag/v1.0-gtzan))
e stima il **BPM** di un file audio. Non serve scaricare nessun dataset.

Passi:
0. dipendenze e import;
1. clone del repo + download dei pesi da GitHub;
2. caricamento del modello (`BPMExtractor`: finestre da 10 s, mediana, TTA +-1 semitono);
3. **demo autocontenuta** su loop sintetici a BPM *noto* (nessun audio di terzi);
4. stima del BPM di un **tuo file** (upload) + grafici (mel, BPM per finestra,
   distribuzione del modello sui bin di BPM).

**Prestazioni del modello rilasciato** (100 tracce di test GTZAN, split senza
leakage): MAE 15.8 BPM, errore **mediano 4.5 BPM**, Acc@1BPM 17%, Acc@5% 55%,
accuratezza octave-tollerante 53%, errori d'ottava 17%, P-score 0.485.
Su materiale 4/4 chiaro (disco/rock/pop/hiphop) l'errore mediano scende sotto i
3 BPM; su jazz/classica/metal anche il riferimento annotato e' ambiguo
(dettagli e limiti nel README del repo).

In [ ]:
#@title 0. Dipendenze e import
%pip install -q torch librosa soundfile matplotlib requests

import os
import sys
from pathlib import Path

import numpy as np
import torch

device = ("cuda" if torch.cuda.is_available()
          else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else "cpu")
print("python", sys.version.split()[0], "| torch", torch.__version__, "| device:", device)

In [ ]:
#@title 1. Clona il repo e scarica i pesi da GitHub
import json, os, subprocess, sys, urllib.request
from pathlib import Path

import numpy as np
import torch

REPO_URL = "https://github.com/Alessandro1040/bpm-crnn.git"
ROOT = Path("/content/bpm-crnn") if Path("/content").is_dir() else Path.cwd() / "bpm-crnn"
if not (ROOT / "src" / "infer.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

CKPT = ROOT / "models" / "bpm_crnn_gtzan.pt"
RAW_URL = "https://raw.githubusercontent.com/Alessandro1040/bpm-crnn/main/models/bpm_crnn_gtzan.pt"
RELEASE_URL = "https://github.com/Alessandro1040/bpm-crnn/releases/download/v1.0-gtzan/best.pth"
if not CKPT.exists():
    try:
        print("scarico i pesi da GitHub (models/bpm_crnn_gtzan.pt, 21 MB) ...")
        urllib.request.urlretrieve(RAW_URL, CKPT)
    except Exception as exc:                       # fallback: release asset (64 MB, con optimizer)
        print("fallback sulla release:", exc)
        urllib.request.urlretrieve(RELEASE_URL, CKPT)
print("pesi:", CKPT, f"{CKPT.stat().st_size/1e6:.1f} MB")

In [ ]:
#@title 2. Carica il modello
from src.infer import BPMExtractor
from src.model import build_model_from_config
from src.utils import load_checkpoint

ckpt = load_checkpoint(CKPT, map_location="cpu")
model = build_model_from_config(ckpt.get("arch") or {})
model.load_state_dict(ckpt["state_dict"])
model.eval()

print("checkpoint :", CKPT.name, "| epoca migliore:", ckpt.get("epoch"))
print("note       :", ckpt.get("note"))
print("architettura:", ckpt.get("arch"))
print(f"parametri  : {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

tm = ckpt.get("test_metrics")
if tm:
    print(f"\nmetriche di test salvate nei pesi (100 tracce GTZAN):")
    print(f"  MAE {tm['mean/MAE']:.2f} BPM | errore mediano {tm['mean/MedianAE']:.2f} BPM | "
          f"Acc@5% {100*tm['mean/Acc_5%']:.1f}% | "
          f"Acc octave@4% {100*tm['mean/Acc_octave_4%']:.1f}% | "
          f"errori d'ottava {100*tm['mean/Octave_error_rate_4%']:.1f}%")

extractor = BPMExtractor(CKPT, device="auto", cache_dir=None)   # sr/n_fft/hop presi dal checkpoint
print("\nspettrogramma:", f"sr={extractor.sr}", f"n_mels={extractor.n_mels}",
      f"n_fft={extractor.n_fft}", f"hop={extractor.hop_length}", f"fmax={extractor.fmax:.0f}")

In [ ]:
#@title 3. Prova su loop sintetici a BPM noto
import soundfile as sf

from src.dataset import synthesize_rhythm

BPM_VERI = [90.0, 120.0, 128.0, 174.0]
for bpm in BPM_VERI:
    y = synthesize_rhythm(bpm, sr=22050, duration=12.0,
                          rng=np.random.default_rng(int(bpm)))
    sf.write(f"/tmp/loop_{int(bpm)}.wav", y, 22050)

print("Loop percussivi sintetici con BPM esatto (nessun audio di terzi).\n")
print(f"{'BPM vero':>9} {'BPM stimato':>12} {'errore':>8} {'finestre':>9} {'spread':>8}")
print("-" * 52)
for bpm in BPM_VERI:
    det = extractor.predict(f"/tmp/loop_{int(bpm)}.wav", return_details=True)
    print(f"{bpm:9.0f} {det['bpm']:12.2f} {det['bpm'] - bpm:+8.2f} "
          f"{det['n_chunks']:9d} {det['bpm_std']:8.2f}")

In [ ]:
#@title 4. Stima il BPM di un tuo file
from google.colab import files

print("Carica un file audio (mp3 / wav / m4a / flac / ogg):")
uploaded = files.upload()
nome = list(uploaded.keys())[0] if uploaded else "/tmp/loop_128.wav"

det = extractor.predict(nome, chunk_duration=10.0, overlap=5.0, tta=True,
                        return_details=True)
print(f"\n{nome}: {det['bpm']:.2f} BPM")
print(f"  mediana su {det['n_chunks']} finestre da {det['chunk_duration']:.0f} s "
      f"(overlap {det['overlap']:.0f} s), TTA={'si' if det['tta'] else 'no'}")
print(f"  spread fra finestre: {det['bpm_std']:.2f} BPM "
      f"(min {det['bpm_min']:.1f} / max {det['bpm_max']:.1f})")
print("  BPM per finestra:", ", ".join(f"{v:.1f}" for v in det["chunk_bpms"]))

In [ ]:
#@title 5. Grafici: mel, BPM per finestra, distribuzione
import librosa
import matplotlib.pyplot as plt
import torch.nn.functional as F
from IPython.display import Audio, display

percorso = nome if "nome" in dir() else "/tmp/loop_128.wav"
det = extractor.predict(percorso, return_details=True)

y, _ = librosa.load(percorso, sr=extractor.sr, mono=True)
mel = librosa.feature.melspectrogram(y=y, sr=extractor.sr, n_mels=extractor.n_mels,
                                     n_fft=extractor.n_fft,
                                     hop_length=extractor.hop_length,
                                     fmax=extractor.fmax)
mel_db = librosa.power_to_db(mel, ref=np.max)

with torch.no_grad():
    out = extractor.model(extractor._mel_tensor(y), return_all=True)
    probs = F.softmax(out["song_logits"], dim=-1).cpu().numpy()[0]
bins = extractor.model.bins.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(mel_db, origin="lower", aspect="auto", cmap="magma")
axes[0].set_title("Mel (dB)"); axes[0].set_xlabel("frame (hop 512)"); axes[0].set_ylabel("bin mel")

axes[1].plot(range(1, det["n_chunks"] + 1), det["chunk_bpms"], "o-")
axes[1].axhline(det["bpm"], ls="--", color="k", label=f"mediana {det['bpm']:.1f} BPM")
axes[1].set_title("BPM per finestra"); axes[1].set_xlabel("finestra")
axes[1].set_ylabel("BPM"); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(bins, probs)
axes[2].axvline(det["bpm"], ls="--", color="k")
axes[2].set_xlim(max(30, det["bpm"] - 40), min(285, det["bpm"] + 40))
axes[2].set_title("Distribuzione del modello sui bin di BPM")
axes[2].set_xlabel("BPM"); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

top = np.argsort(probs)[::-1][:3]
print("top-3 del modello:",
      ", ".join(f"{bins[i]:.0f} BPM ({probs[i]:.1%})" for i in top))
display(Audio(percorso))

### Note e limiti

* La stima finale e' la **mediana** delle finestre: robusta ai cambi di tempo e alle
  finestre "sfortunate"; `bpm_std` basso (~0.5-1 BPM) indica un brano con pulsazione
  chiara, alto (10-20 BPM) un brano ambiguo (e spesso sbagliato).
* Il modello e' addestrato su **10 s** di spettrogramma Mel: per musica 4/4 "dritta"
  finestre da 30 s (`chunk_duration=30, overlap=0`) migliorano le metriche
  (MAE 14.8 invece di 15.8 sul test set).
* Errori tipici: **jazz, classica, metal, country** (il riferimento annotato e' discusso)
  e materiale senza percussioni evidenti. Gli errori d'ottava (mezzo/doppio tempo)
  sono solo il 17 % del totale.
* TTA (`tta=True`) valuta ogni finestra anche a +-1 semitono: piu' robusto al timbro,
  ~3x piu' lento. Disattivalo con `tta=False` se hai fretta.